# DC3 – Arctic Sea Ice Forecasting: Evaluation Quickstart

This notebook demonstrates the complete DC3 evaluation workflow:

1. **Generate** a sample submission (random noise on the EPSG:3413 polar grid)
2. **Inspect** the dataset structure
3. **Validate** the format against the DC3 specification
4. **Run** the evaluation pipeline
5. **Visualise** the results

> **DC3 is a sea-ice forecasting challenge.** Predictions cover the Arctic on a
> polar stereographic grid (EPSG:3413, ~3.25 km resolution) and span **181 daily
> lead times** (one full Arctic winter–spring season).

## 0. Setup

Ensure the `dc-env` conda environment is activated before running this notebook:

```bash
conda activate dc-env
jupyter lab notebooks/evaluation_quickstart.ipynb
```

Or install the package manually:

```bash
pip install -e .
pip install "dctools @ git+https://github.com/ocean-ai-data-challenges/dc-tools.git"
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import xarray as xr

# Ensure the project root is on the Python path (works from notebooks/)
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 1. Generate a sample submission

DC3 predictions are **sea ice concentration** (`siconc`) fields on the EPSG:3413
Arctic polar stereographic grid.

Each forecast file must cover **181 daily lead times** (0–180 days) starting from
an initialisation date in autumn, with auxiliary 2-D `lat`/`lon` arrays.

The cell below generates a minimal random-noise dataset that conforms to the DC3
specification — useful to test the full pipeline before plugging in your model.

In [ ]:
import pandas as pd

# ── DC3 grid specification (EPSG:3413 Arctic polar stereographic) ─────
DC3_X = np.arange(-3_850_000, 3_750_001, 3_250, dtype="float32")   # ~2 338 pts
DC3_Y = np.arange(-5_350_000, 5_850_001, 3_250, dtype="float32")   # ~3 447 pts
N_LEAD_TIMES = 181  # full seasonal forecast (0..180 days)

OUTPUT_DIR = Path("/tmp/dc3_quickstart_sample")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

# Build auxiliary 2-D lat/lon grids (approximate, for illustration)
xx, yy = np.meshgrid(DC3_X, DC3_Y, indexing="ij")
# Rough geographic approximation: not projection-exact, just shape-correct
lat_2d = np.clip(90.0 - np.hypot(xx, yy) / 111_320.0, -90, 90).astype("float32")
lon_2d = (np.degrees(np.arctan2(xx, yy)) % 360).astype("float32")

# Generate one forecast Zarr store covering the full 181-day window
siconc_data = np.clip(
    rng.random((N_LEAD_TIMES, len(DC3_X), len(DC3_Y)), dtype="float32"),
    0.0, 1.0,
)

times = pd.date_range("2020-11-01", periods=N_LEAD_TIMES, freq="1D")

ds = xr.Dataset(
    {
        "siconc": xr.DataArray(siconc_data, dims=["time", "x", "y"]),
        "lat":    xr.DataArray(lat_2d, dims=["x", "y"]),
        "lon":    xr.DataArray(lon_2d, dims=["x", "y"]),
    },
    coords={
        "time": times,
        "x":    DC3_X,
        "y":    DC3_Y,
    },
)
ds["siconc"].attrs["long_name"] = "sea ice area fraction"
ds["siconc"].attrs["units"]     = "1"

store_path = OUTPUT_DIR / "siconc_2020-2021.zarr"
ds.to_zarr(str(store_path), mode="w", consolidated=True)

print(f"Sample forecast written to: {store_path}")
print(f"  siconc shape: {siconc_data.shape}  (time × x × y)")
print(f"  time:  {times[0].date()} → {times[-1].date()}  ({len(times)} days)")
print(f"  x:     {DC3_X[0]/1e6:.2f} to {DC3_X[-1]/1e6:.2f} Mm  ({len(DC3_X)} pts)")
print(f"  y:     {DC3_Y[0]/1e6:.2f} to {DC3_Y[-1]/1e6:.2f} Mm  ({len(DC3_Y)} pts)")

## 2. Inspect the dataset

Open the generated Zarr store and verify its structure matches the DC3 specification.

In [ ]:
ds_check = xr.open_zarr(str(store_path))

print(f"x:     {ds_check.x.values[0]:.0f} m  →  {ds_check.x.values[-1]:.0f} m  ({len(ds_check.x)} pts)")
print(f"y:     {ds_check.y.values[0]:.0f} m  →  {ds_check.y.values[-1]:.0f} m  ({len(ds_check.y)} pts)")
print(f"time:  {str(ds_check.time.values[0])[:10]}  →  {str(ds_check.time.values[-1])[:10]}  ({len(ds_check.time)} timesteps)")
print(f"vars:  {list(ds_check.data_vars)}")

ds_check

## 3. Quick conformity check

Verify that the submission meets the DC3 grid specification before running the
full (expensive) evaluation.

In [ ]:
# ── DC3 specification constants ────────────────────────────────────────
_DC3_X_START  = -3_850_000.0
_DC3_X_STOP   =  3_750_000.0
_DC3_Y_START  = -5_350_000.0
_DC3_Y_STOP   =  5_850_000.0
_DC3_STEP     =  3_250.0
_DC3_START    = pd.Timestamp("2020-11-01")
_DC3_END      = pd.Timestamp("2021-05-01")
_N_LEAD_TIMES = 181
_MAX_NAN_FRAC = 0.10

issues = []

# Variable
if "siconc" not in ds_check.data_vars:
    issues.append("❌ Missing variable: siconc")
else:
    print("✅ Variable 'siconc' present.")

# x/y grid
for dim, start, stop in (("x", _DC3_X_START, _DC3_X_STOP), ("y", _DC3_Y_START, _DC3_Y_STOP)):
    if dim not in ds_check.dims:
        issues.append(f"❌ Missing dimension: {dim}")
    else:
        v = ds_check[dim].values
        if not np.isclose(v[0], start, atol=_DC3_STEP):
            issues.append(f"❌ {dim} start: expected ~{start:.0f}, got {v[0]:.0f}")
        elif not np.isclose(v[-1], stop, atol=_DC3_STEP):
            issues.append(f"❌ {dim} stop: expected ~{stop:.0f}, got {v[-1]:.0f}")
        else:
            print(f"✅ {dim} grid matches DC3 specification ({len(v)} pts).")

# Temporal coverage
n_times = len(ds_check.time)
if n_times < _N_LEAD_TIMES:
    issues.append(f"❌ Time dimension: expected {_N_LEAD_TIMES} lead times, got {n_times}.")
else:
    print(f"✅ Time dimension has {n_times} lead times (≥ {_N_LEAD_TIMES} required).")

# Auxiliary lat/lon
for aux in ("lat", "lon"):
    if aux not in ds_check.data_vars and aux not in ds_check.coords:
        issues.append(f"⚠️  Auxiliary coordinate '{aux}' not found (needed for per-bin maps).")
    else:
        print(f"✅ Auxiliary coordinate '{aux}' present.")

# NaN fraction
nan_frac = float(ds_check["siconc"].isnull().mean().values)
status   = "✅" if nan_frac <= _MAX_NAN_FRAC else "❌"
print(f"{status} siconc NaN fraction = {nan_frac:.2%}")
if nan_frac > _MAX_NAN_FRAC:
    issues.append(f"❌ siconc NaN fraction {nan_frac:.1%} exceeds threshold {_MAX_NAN_FRAC:.0%}.")

if issues:
    print(f"\n━━ {len(issues)} issue(s) found ━━")
    for issue in issues:
        print(f"  {issue}")
else:
    print("\n✅ All checks passed — submission is DC3-compliant!")

## 4. Visualise the prediction field

Plot the sea ice concentration at the first and last lead times to get a sense
of the spatial structure.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, t_idx, label in zip(axes, [0, -1], ["Lead time = day 0", "Lead time = day 180"]):
    img = ds_check["siconc"].isel(time=t_idx).values.T   # transpose for (y, x) display
    im  = ax.imshow(
        img,
        origin="lower",
        cmap="Blues",
        vmin=0, vmax=1,
        aspect="auto",
    )
    ax.set_title(f"{label}\nDate: {str(ds_check.time.values[t_idx])[:10]}")
    ax.set_xlabel("x index")
    ax.set_ylabel("y index")
    plt.colorbar(im, ax=ax, label="siconc (0–1)", shrink=0.7)

fig.suptitle("DC3 sample forecast – sea ice concentration (siconc)")
plt.tight_layout()
plt.show()

## 5. Visualise existing results (TOPAZ4 baseline)

If a full evaluation has already been run, load and visualise the RMSD scores.
DC3 evaluates sea ice concentration predictions against **AMSR2** (primary),
**IABP** (in-situ buoy temperatures), and **MODIS** (lead maps).

In [ ]:
# Try dc3_output/results first, then fall back to leaderboard_results
results_path = PROJECT_ROOT / "dc3_output" / "results" / "results_topaz.json"
if not results_path.exists():
    results_path = PROJECT_ROOT / "dc3" / "leaderboard_results" / "results_topaz.json"

if results_path.exists():
    with open(results_path) as f:
        data = json.load(f)
    model_name = data["dataset"]
    records = data["results"][model_name]
    print(f"Loaded results for model: {model_name}  ({len(records)} evaluation records)")
    print(f"Reference datasets: {sorted(set(r['ref_alias'] for r in records))}")
else:
    print("No results file found.")
    print("Run the full evaluation first:")
    print("  python dc3/evaluate.py --model-name topaz")
    records = []
    model_name = "topaz"

In [ ]:
# Parse RMSD scores grouped by reference dataset and lead time
scores = {}  # {ref_alias: {variable: {lead_time: value}}}

for record in records:
    ref = record["ref_alias"]
    lt  = record.get("lead_time")
    for metric in record["result"]:
        if metric["Metric"] not in ("rmse", "rmsd"):
            continue
        var = metric["Variable"]
        scores.setdefault(ref, {}).setdefault(var, {})[lt] = metric["Value"]

# RMSD vs lead time for AMSR2 reference (primary leaderboard metric)
if "amsr2" in scores:
    amsr2 = scores["amsr2"]
    fig, ax = plt.subplots(figsize=(11, 4))
    for var_name in sorted(amsr2):
        lts  = sorted(amsr2[var_name].keys())
        vals = [amsr2[var_name][lt] for lt in lts]
        ax.plot(lts, vals, marker=".", label=var_name)
    ax.set_xlabel("Lead time (days)")
    ax.set_ylabel("RMSD")
    ax.set_title(f"{model_name} – Sea Ice Concentration RMSD vs AMSR2")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No AMSR2 results to plot. Run the evaluation first.")

In [ ]:
# Compare siconc RMSD across all reference datasets
sic_refs = [ref for ref in scores if any("siconc" in v or "z" == v for v in scores[ref])]

if len(sic_refs) > 1:
    fig, ax = plt.subplots(figsize=(11, 4))
    for ref in sorted(sic_refs):
        for var_name in scores[ref]:
            lts  = sorted(scores[ref][var_name].keys())
            vals = [scores[ref][var_name][lt] for lt in lts]
            ax.plot(lts, vals, marker=".", label=f"{ref} / {var_name}")
    ax.set_xlabel("Lead time (days)")
    ax.set_ylabel("RMSD")
    ax.set_title(f"{model_name} – Sea Ice RMSD by reference dataset")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
elif not scores:
    print("No scores available. Run the evaluation first.")

## Next steps

- Replace the random-noise sample with your model's sea ice concentration predictions
- Run the full evaluation:
  ```bash
  python dc3/evaluate.py --model-name MyModel
  ```
- Submit results for the [leaderboard](https://github.com/ppr-ocean-ia/dc3-sea-ice-forecasting)
- See the [full submission workflow](submit.ipynb) notebook for a step-by-step guide

See the [full documentation](docs/source/index.md) for details.